# Setup

In [ ]:
import sys
project_root = "/data/workdata/709656/Anne/Emilys_code/"
sys.path.insert(0, project_root)

from split_format import split_and_format_data
from xgboost_model import train_xgboost_model_random

In [ ]:
DATA_PATH = ("/Data_files/cohort_data/cohort_50_to_54.parquet" )
RANDOM_STATE=42
MODEL_NAME="Model5"
MAXIMIZE_METRIC = "f2" # or pr_auc, recall, precision, roc_auc
#MIN_PRECISION =0.15

In [ ]:
from pathlib import Path
COHORT_NAME= "male_50-54"
OUTPUT_DIR = Path(
    f"/XBoost_results/{COHORT_NAME}"
)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

# Data Loading

In [ ]:
import fastparquet
# Load, split, preprocess
(
X_train_raw,
X_cal_raw,
X_test_raw,
y_train,
y_cal,
y_test,
id_fit,
if_cal,
id_test,
preprocessor
) = split_and_format_data(
    data_path=DATA_PATH,
    drop_cols=[
        "pnr",
        "family_id",
        "in_dk",
        "de_age",
        "alive",
        "de_parish",
        "de_region",
        "de_municipality",
        "de_time_to_death",
        "de_age_at_death",
        "se_educ_date",
        "de_sex",
    ],
sex_filter = ["Male"],
target_col="early_death",
test_size=0.3,
cal_size_within_train=0,
random_state=RANDOM_STATE,
stratify_on_year=True,
year_col="year",
)
print("Train shape:", X_train_raw.shape)
print("Test shape:", X_test_raw.shape)
print("Train death rate:", y_train.mean())
print("Test death rate:", y_test.mean())

In [ ]:
# save death rates for later
import numpy as np
raw_train_death_rate = float(np.mean(y_train))
test_death_rate =float(np.mean(y_test))

# Model Training

In [ ]:
#Scaleposweight
spw= (1-y_train.mean()) /y_train.mean()
print(spw)

In [ ]:
#Hyperparameters
param_grid = {
    "max_depth": [2,3,4],
    "learning_rate": [0.01,0.03,0.05],
    "n_estimators": [400,800,1000],
    "subsample": [0.5, 0.6, 0.7],
    "colsample_bytree": [0.3, 0.5, 0.7],
    "gamma":[2,5,10,20],
    "min_child_weight":[10,20,40],
    "reg_lambda":[20,40,80],
    "reg_alpha":[1,5,10],
}

In [ ]:
search_seed = abs(hash(COHORT_NAME)) % (2**32)
model5_best_model, model_5_best_params,model5_thr = train_xgboost_model_random(
    X_train_raw,
    y_train,
    preprocessor=preprocessor,
    param_grid=param_grid,
    cv_folds=3,
    random_state=search_seed,
    maximize="f2",
    #min_precision=0.15,
    scale_pos_weight=spw,
    n_iter=60, )


## Threshold

In [ ]:
#Find optimal threshold maximizing f2
from sklearn.metrics import precision_recall_curve
import numpy as np
y_prob_train = model5_best_model.predict_proba(X_train_raw)[:, 1]
precision, recall, thresholds= precision_recall_curve(y_train,y_prob_train)
# Choose threshold that maximizes f2 on train set
beta=2
p= precision[:-1]
r= recall[:-1]
f2 = (1 + beta**2) * (p *r) / (beta**2 * p + r + 1e-12)
best_idx = np.argmax(f2)
model5_thr = float(thresholds[best_idx])
print("Chosen threshold (train , max f2):", model5_thr)
print("Precision / Recall at chosen thr:", p[best_idx], r[best_idx])
print("F2 at chosen thr:", f2[best_idx])

## Evaluation

In [ ]:
import json
from pathlib import Path
from datetime import datetime
import numpy as np
from sklearn.metrics import(
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    accuracy_score,
    fbeta_score,
    roc_auc_score,
    matthews_corrcoef,
)

In [ ]:
# Evaluation on test set
y_prob=model5_best_model.predict_proba(X_test_raw)[:,1]
y_pred = (y_prob >=model5_thr).astype(int)
test_f1= f1_score(y_test, y_pred)
test_f2 =fbeta_score(y_test,y_pred,beta=2)
test_prec=precision_score(y_test,y_pred)
test_rec=recall_score(y_test,y_pred)
test_roc=roc_auc_score(y_test,y_prob)
test_pr=average_precision_score(y_test,y_prob)
test_bal=balanced_accuracy_score(y_test,y_pred)
test_acc=accuracy_score(y_test,y_pred)
test_mcc= matthews_corrcoef(y_test, y_pred)
print("\n ==== Test metric=====" )
print("F1 score:", test_f1)
print("F2 score:", test_f2)
print("Precision:", test_prec)
print("Recall:", test_rec)
print("ROC-AUC", test_roc)
print("PR-AUC (avg prec):", test_pr)
print("Balanced accuracy:", test_bal)
print("Accuracy:", test_acc)
print("MCC:", test_mcc),

In [ ]:
# Evaluation on train set
for col in X_train_raw.select_dtypes(include="object").columns:
    X_train_raw[col] = X_train_raw[col].astype("category")
y_prob_train=model5_best_model.predict_proba(X_train_raw)[:,1]
y_pred_train = (y_prob_train >=model5_thr).astype(int)
train_f1= f1_score(y_train, y_pred_train)
train_f2 =fbeta_score(y_train,y_pred_train,beta=2)
train_prec=precision_score(y_train,y_pred_train)
train_rec=recall_score(y_train,y_pred_train)
train_roc=roc_auc_score(y_train,y_prob_train)
train_pr=average_precision_score(y_train,y_prob_train)
train_bal=balanced_accuracy_score(y_train,y_pred_train)
train_acc=accuracy_score(y_train,y_pred_train)
train_mcc= matthews_corrcoef(y_train, y_pred_train)
print("\n ==== Raw train metric=====" )
print("F1 score:", train_f1)
print("F2 score:", train_f2)
print("Precision:", train_prec)
print("Recall:", train_rec)
print("ROC-AUC", train_roc)
print("PR-AUC (avg prec):", train_pr)
print("Balanced accuracy:", train_bal)
print("Accuracy:", train_acc)
print("MCC:", train_mcc)

In [ ]:
# Confusion matrix on test set 
cm = confusion_matrix(y_test,y_pred)
tn,fp,fn,tp=cm.ravel()
specificity =tn/(tn+fp)
print("\n === Confusion matrix (test set) === ")
print(cm)
print(f"TN:{tn}, FP: {fp}, FN: {fn}, TP:{tp}")
print("Specificity (TNR)", specificity)

In [ ]:
# ROC-AUC kurve.
import matplotlib as plt
from sklearn.metrics import roc_curve, auc
#ROC curve
fpr,tpr, thresholds=roc_curve(y_test,y_prob)
roc_auc = auc(fpr,tpr)
plt.figure()
plt.plot(fpr,tpr, label=f"ROC curve (AUC={roc_auc:.3f} )")
plt.plot([0,1], [0,1], linestyle="--", label="Random")
plt.xlabel("False Postive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.title(f"ROC curve, testdata, {COHORT_NAME}")
plt.savefig(OUTPUT_DIR/ f"roc_curve_{COHORT_NAME}.png", dpi=300, bbox_inches="tight") 
plt.show()

## Precision recall curves

In [ ]:


y_prob=model5_best_model.predict_proba(X_test_raw)[:,1]
from sklearn.metrics import precision_recall_curve
precision, recall,thresholds= precision_recall_curve(y_test,y_prob)

In [ ]:
thr_idx=np.argmin(np.abs(thresholds-model5_thr))
rec_at_thr=recall[ thr_idx+1]
pr_at_thr=precision[thr_idx+1]

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plt.figure(figsize=(7, 5))

plt.plot(recall, precision, label="Precision-recall curve")

plt.scatter(
    rec_at_thr,
    pr_at_thr,
    s=80,
    zorder=5,
    color="red"
)

plt.annotate(
    f"Recall={rec_at_thr:.2f}, Precision={pr_at_thr:.2f}, Thr={model5_thr:.2f}",
    xy=(rec_at_thr, pr_at_thr),
    xytext=(rec_at_thr + 0.05, pr_at_thr + 0.05),
    textcoords="data",
    arrowprops=dict(arrowstyle="->", color="black"),
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.9)
)

plt.axvline(
    x=rec_at_thr,
    linestyle="--",
    linewidth=1.5,
    color="red",
    alpha=0.9
)

plt.axhline(
    y=pr_at_thr,
    linestyle="--",
    linewidth=1.5,
    color="red",
    alpha=0.9
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall curve on test data for {COHORT_NAME}")
plt.legend(loc="upper right")
plt.grid()

plt.savefig(
    OUTPUT_DIR / f"precision_recall_curve_{COHORT_NAME}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
precision, recall, threshold = precision_recall_curve(y_test,y_prob)
pr_auc=average_precision_score(y_test,y_prob)
plt.plot(threshold,precision[:-1], label="Precision")
plt.plot(threshold,recall[:-1], label= "Recall")
plt.axvline(
    x=model5_thr,
    color="red",
        linestyle="--",
    linewidth=2,
    #label=f"Chosen threshold ={model5_thr:.2f} "
)
plt.scatter(model5_thr, pr_at_thr, color="red",s=100,zorder=6)
plt.scatter(model5_thr, rec_at_thr, color="red",s=100,zorder=6)
plt.hlines(
    y=pr_at_thr,
    xmin=-0,
    xmax=model5_thr,
    colors="red",
    linestyles="dashed",
    linewidth=2
)
plt.hlines(
    y=rec_at_thr,
    xmin=0,
    xmax=model5_thr,
    colors="red",
    linestyles="dashed",
    linewidth=2
)
textstr= (
    f"Threshold={model5_thr:.2f}\n"
    f"Recall={rec_at_thr:.2f}\n"
    f"Precision={pr_at_thr:.2f}"
)
plt.text(
    0.6,0.8,
    textstr,
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.3",
        facecolor="white",
        edgecolor="black",
        alpha=0.9
)
)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend(loc="center left", frameon=True)
plt.grid()
plt.title(f"Precision, recall vs threshold, testdata,{COHORT_NAME}")
plt.savefig(OUTPUT_DIR/ f"precision_recall_threshold_{COHORT_NAME}.png",
dpi=300, bbox_inches="tight") 
plt.show

In [ ]:
## Predicted probability for survivors vs deaths + gap
    # split data by outcome
X_survivors = X_test_raw[y_test == 0].copy()
X_deaths = X_test_raw[y_test == 1].copy()

In [ ]:
# model
    #  gap
p_survivors = model5_best_model.predict_proba(X_survivors)[:, 1].mean()
p_deaths = model5_best_model.predict_proba(X_deaths)[:, 1].mean()
total_gap = p_deaths - p_survivors
print("=== Predicted probabilites on test ===")
print(f"Survivors (y=0), mean p(death): {p_survivors:.4f}")
print(f"Early deaths (y=1), mean p(death): {p_deaths:.4f}")
print(f"Mortality gap (death-survivors): {total_gap:.4f}")

predicted_mortality_rate =float(y_prob.mean())
print(predicted_mortality_rate)

# Save Results

In [ ]:
from pathlib import Path
import pandas as pd
#Save model as json
model5_best_model.named_steps["model"].save_model(
    OUTPUT_DIR /f"model5_best_model_{COHORT_NAME}.json"
)

In [ ]:
from pathlib import Path
import pandas as pd
# Save X test and y test after preprocessing
preprocessor_fitted =model5_best_model.named_steps["preprocess"]
X_test_processed=preprocessor_fitted.transform(X_test_raw)
if hasattr(X_test_processed,"toarray"): X_test_processed= X_test_processed.toarray()
X_test_processed=pd.DataFrame(
    X_test_processed,
    columns=preprocessor_fitted.get_feature_names_out()
)
X_test_processed.columns = (
    X_test_processed.columns
    .str.replace("^remainder__","", regex=True)
    .str.replace("^cat__","", regex=True)
)
X_test_processed.to_csv(OUTPUT_DIR / f"X_test_{COHORT_NAME}.csv", index=False)

pd.DataFrame({"y_test": y_test}).to_csv(
    OUTPUT_DIR / f"y_test_{COHORT_NAME}.csv", index=False)


In [ ]:
# Save predictions
from pathlib import Path
import pandas as pd
import numpy as np
y_prob=model5_best_model.predict_proba(X_test_raw)[:,1]
y_pred = (y_prob >=model5_thr).astype(int)
pred_df = pd.DataFrame({
    "pnr":id_test.astype(str),
    "y_test":np.asarray(y_test).astype(int),
    "y_proba":y_prob,
    "y_pred":y_pred,
    "threshold":model5_thr,
    "test_death_rate":y_test.mean()
})
pred_df.to_parquet(OUTPUT_DIR/ "predictions.parquet", index=False)

In [ ]:
X_test_raw.to_parquet(
    OUTPUT_DIR/ f"X_test_raw_{COHORT_NAME}.parquet", index=False
)
pd.DataFrame({"y_test": y_test}).to_parquet(OUTPUT_DIR / f"y_test_raw_{COHORT_NAME}.parquet",
    index=False
)


In [ ]:
# Bootstrap confidence intervals for the gap in predicted probabilities of survivors vs early death
import numpy as np
import pandas as pd
def bootstrap_gap(y_test, y_prob, n_bootstrap=2000, random_state=42): 
    rng=np.random.default_rng(random_state)

    y_test=np.asarray(y_test)
    y_prob=np.asarray(y_prob)
    # Separate predictions by outcome
    p_surv = y_prob[y_test==0]
    p_death = y_prob[y_test==1]
    # observed gap
    gap_obs = p_death.mean() - p_surv.mean()
    gaps = []
    for i in range(n_bootstrap):
        # stratified resampling
        surv_sample = rng.choice(p_surv, size=len(p_surv), replace=True)
        death_sample = rng.choice(p_death, size=len(p_death), replace=True)

        gap = death_sample.mean() - surv_sample.mean()
        gaps.append(gap)

    gaps = np.array(gaps)
    ci_low = np.percentile(gaps, 2.5)
    ci_up = np.percentile(gaps, 97.5)
    return gap_obs, ci_low, ci_up

# Compute on the test set
gap, ci_low, ci_up = bootstrap_gap(y_test, y_prob, n_bootstrap=2000)
print(f"Gap (death - survivor) = {gap:.4f}")
print(f"95% CI: [{ci_low:.4f}, {ci_up:.4f}]")

In [ ]:
chosen_threshold=model5_thr

In [ ]:
# Save results

# pyrefly: ignore
# ruff: noqa

import json
import numpy as np
from datetime import datetime

time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

results = {
    "model_name": "Model5",

    "objective": {
        "cohort": f"{COHORT_NAME}",
        "maximize": "f2",
        "eval_metric": "aucpr",
        "min_precision": 0,
        "cv_folds": 3,
        "n_iter": 60,
        "chosen_threshold": float(model5_thr),
        "scale_pos_weight": float(spw),
        # "threshold_source": "OOF_train",
        "comments": "Class imbalance handled with scale_pos_weight instead of rebalancing"
    },

    "best_params": model_5_best_params,

    "death_rate_summary": {
        "raw_train_death_rate": float(raw_train_death_rate),
        "test_death_rate": float(test_death_rate),
        "calibration_death_rate": float(cal_death_rate),
        # "resampled_train_death_rate": float(resampled_train_death_rate),
    },

    "test_metrics": {
        "f1": float(test_f1),
        "f2": float(test_f2),
        "precision": float(test_prec),
        "recall": float(test_rec),
        "roc_auc": float(test_roc),
        "pr_auc": float(test_pr),
        "specificity_TNR": float(specificity),
        "balanced_accuracy": float(test_bal),
        "accuracy": float(test_acc),
        "MCC": float(test_mcc),
        "threshold_used": float(chosen_threshold),
    },

    "train_metrics": {
        "f1": float(train_f1),
        "f2": float(train_f2),
        "precision": float(train_prec),
        "recall": float(train_rec),
        "roc_auc": float(train_roc),
        "pr_auc": float(train_pr),
        "balanced_accuracy": float(train_bal),
        "accuracy": float(train_acc),
        "MCC": float(train_mcc),
        "threshold_used": float(model5_thr),
    },

    "confusion_matrix_test": {
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    },

    "predicted_mortality_rate_test": {
        "overall_predicted_mortality_rate": float(predicted_mortality_rate),
    },

 "predicted_probabilites test":{
     "mean_survived":float(np.mean(y_prob[y_test==0])),
     "mean_died":float(np.mean(y_prob[y_test==1])),
     "gap_died_minus_survived": float(
     np.mean(y_prob[y_test==1])-np.mean(y_prob[y_test==0])),
             "gap_ci_95_boostrap": {
            "lower": float(ci_low),
            "upper":float(ci_up)
} },
    }


out_path = OUTPUT_DIR / f"Model_5_{COHORT_NAME}_{time_stamp}.json"

with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved JSON:", out_path)